In [2]:
import os
from dotenv import load_dotenv

In [57]:
load_dotenv()
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

In [5]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
import requests

In [58]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()

@tool
def get_weather_data(city: str) -> str:
  """
  This function fetches the current weather data for a given city
  """
  url = f'https://api.weatherstack.com/current?access_key={WEATHERSTACK_API_KEY}&query={city}'

  response = requests.get(url)

  return response.json()

results = search_tool.invoke('top news in maharashtra about linkroad')

In [59]:
results

'4 May 2026 ... ... Road safety enforcement on Expressway, Latest Marathi news on Mumbai Pune road rules] ... MTN MAHARASHTRA TRENDING NEWS प्रवाशांसाठी आनंदाची बातमी! 2 May 2026 ... Important news has emerged for travelers commuting between Mumbai and Pune on the expressway. The Maharashtra State Road Development Corporation (MSRDC) has ... 1 May 2026 ... ... road project news maharashtra, highway connectivity update #mumbaipune ... TOP NEWS MARATHI : Connecting link resumed traffic: Mumbai Pune expressway ... 8 Apr 2026 ... ... News #Kandivali #LinkRoad #DahanukarwadiMetro #MumbaiTraffic #TrafficJam #MumbaiNews #KandivaliLinkRoad. ... Go to channel Lokshahi Marathi · TOP 50 Fast News | ... 1 May 2026 ... ... maharashtra breaking news, marathi news today,marathi batmya,batmya marathi,live news marathi,times now marathi,maharashtra politics,political crisis'

In [60]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.2
)

In [61]:
llm.invoke('hii')

AIMessage(content='Hi there! How can I help you today?', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f7e54-d01d-7901-b511-b62d82e79862-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 2, 'output_tokens': 31, 'total_tokens': 33, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 21}})

In [62]:
from langchain_classic.agents import AgentExecutor,create_react_agent
from langchain_classic import hub

In [63]:
from langsmith import Client

client = Client()

prompt = client.pull_prompt("hwchase17/react")

In [64]:
agent = create_react_agent(
    llm=llm,
    tools=[search_tool,get_weather_data],
    prompt=prompt
)

In [65]:
agent_executor = AgentExecutor(
    agent=agent,
    tools=[search_tool,get_weather_data],
    verbose=True
)

In [67]:
response = agent_executor.invoke({'input':"Find the capital of Madhya Pradesh, then find it's current weather condition"})



> Entering new AgentExecutor chain...
Action: duckduckgo_search
Action Input: capital of Madhya PradeshIndore (/ ɪnˈdɔːr / ⓘ; ISO: Indaura, Hindi: [ɪn̪d̪ɔːr]) is the largest city of the Indian state of Madhya Pradesh where it serves as the capital and the administrative headquarters of the eponymous district and division. Modern-day Indore was established on the banks of the Kanh and Saraswati rivers and traces its roots to its 16th-century founding as a trading hub between the ... Gwalior was a premier twenty-one Gun Salute State along with Hyderabad, Mysore, Jammu and Kashmir and Baroda. [9] Post Independence, the city became the capital of the short-lived state of Madhya Bharat, now part of the larger state of Madhya Pradesh. Madhya Pradesh, state of India that is situated in the heart of the country. It has no coastline and no international frontier. Its physiography is characterized by low hills, extensive plateaus, and river valleys. The capital is Bhopal, in the west-central p

In [68]:
response['output']

'The capital of Madhya Pradesh is Bhopal. The current weather condition in Bhopal is partly cloudy with a temperature of 28°C, feeling like 34°C. The wind speed is 16 km/h from the West, humidity is 84%, and there is 0.9mm of precipitation.'